# xAI Grok 4.7 on Amazon Bedrock

> **Sample code — not for production.** Provided as AWS Content under the AWS
> Customer Agreement; do not use it in production accounts or on production or other
> critical data. Running these cells calls Amazon Bedrock and incurs charges. Full
> disclaimer in the [README](../README.md#disclaimer).

xAI's Grok 4.7 is built for coding, agentic tasks and knowledge work, with a
524,288-token context window shared between input and output, per its [model
card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-xai-grok-4-7.html).
It reasons on every request, its reasoning is encrypted, and it reads images. Unlike
Grok 4.6 it is served on `bedrock-runtime` only, through a cross-Region inference
profile. Grok 4.6 has its own notebook, [`02-grok-4-6`](02-grok-4-6.ipynb),
and Grok 4.3 [`01-grok-4-3`](01-grok-4-3.ipynb).

| Model | `bedrock-mantle` ID | `bedrock-runtime` ID | Notes |
|---|---|---|---|
| Grok 4.7 | not served | `us.xai.grok-4.7` or `global.xai.grok-4.7` | `bedrock-runtime` only, through an inference profile |

IDs, Regions, window and lifecycle read from the model card on 20 October 2026. Its
Programmatic Access table has one row, `bedrock-runtime` with the model ID
`xai.grok-4.7` and `Not supported` as the In-Region endpoint URL, so an inference
request must name a profile: `us.xai.grok-4.7` from the six source Regions the card's US
geo table lists (`us-east-1`, `us-east-2`, `us-west-1`, `us-west-2`, `ca-central-1` and
`ca-west-1`, and that profile answered in the Canadian ones on 18 October 2026), or
`global.xai.grok-4.7`, which the card marks available in every Region on that list
except `ca-west-1` and the GovCloud pair. The window is `524,288 tokens, shared between
input and output`, and the card puts the output limit at the window minus the input
rather than at a fixed number.

Geo is the only option the card prices in GovCloud: its separate AWS GovCloud (US)
pricing table carries a `Geo CRIS` row and no Global one. Which Geo profile a GovCloud
caller names is not on the card, because its `Geo: US` routing table lists only the six
source Regions above, so `us.xai.grok-4.7` is unconfirmed from `us-gov-east-1` and
`us-gov-west-1`, and no cell here calls from either Region. The card's direction for
that case is to call `GetInferenceProfile` from the source Region for its current
routing configuration, which is what section 11 does with an ID it already holds: that
operation takes `inferenceProfileIdentifier` as a required parameter, so it reports the
destinations of a profile you can already name rather than naming one for you. The call
that reports which profiles a Region carries is `ListInferenceProfiles` with
`typeEquals="SYSTEM_DEFINED"`, which returned `us.xai.grok-4.7` and
`global.xai.grok-4.7`, both `ACTIVE`, from `us-east-1` on 7 November 2026, where
`usgov.xai.grok-4.7`, `us-gov.xai.grok-4.7` and the bare `xai.grok-4.7` each came back
`ResourceNotFoundException: Inference profile not found` from `GetInferenceProfile`. Run
that listing in `us-gov-east-1` or `us-gov-west-1` to read the IDs those Regions carry
before hard-coding a prefix (card read 7 November 2026).

`bedrock-mantle` does not serve it at all, which the card's endpoint table also says:
the bare ID returned 404 `The model 'xai.grok-4.7' does not exist` in `us-east-1` and
`us-west-2` in October 2026, where `xai.grok-4.6` answers on that same path in
`us-west-2`. Grok 4.6's `bedrock-mantle` section and its background mode therefore have
no equivalent here. Grok 4.7 launched on 28 September 2026, so it is the first model in
this folder under the [current lifecycle
policy](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle.html) rather than the [one for models launched before 7
September 2026](https://docs.aws.amazon.com/bedrock/latest/userguide/model-lifecycle-legacy.html) that covers Grok 4.3 and 4.6: the card
reads Active with `Model EOL date: N/A` and an "EOL no sooner than" date of 28 September
2027.

This notebook calls the **Responses** API on `bedrock-runtime`, with the OpenAI SDK at
`https://bedrock-runtime.{region}.amazonaws.com/openai/v1`. Chat Completions,
Converse and InvokeModel are on the same endpoint (sections 8, 9 and 10).

**Covered:** first call, streaming, multi-turn and server-side state, reasoning, tool
use, structured output, vision, Chat Completions, Converse, InvokeModel, IAM
permissions, and prompt caching.

## Setup

Install the pinned requirements first (`pip install -r ../requirements.txt`, Python 3.11
or later). The notebook calls Bedrock with the OpenAI SDK and boto3. From
`_shared/bedrock.py`, this repository's own helper module, it takes a lenient JSON
parser, a print that redacts account IDs, and the sample slide.

Endpoints, authentication and URL paths are explained in
[`00-foundations/01`](../00-foundations/01-endpoints-auth-and-the-three-paths.ipynb).

In [1]:
import json
import re
import sys

import boto3
from aws_bedrock_token_generator import provide_token
import openai
from openai import OpenAI

sys.path.insert(0, "../_shared")
from bedrock import parse_json_lenient, safe_print

REGION = "us-east-1"
GROK = "us.xai.grok-4.7"  # the US geographic inference profile

# bedrock-runtime's OpenAI-compatible path takes the same short-term Bedrock API key as
# bedrock-mantle. Reasoning calls can take minutes, so set a client timeout.
client = OpenAI(
    base_url=f"https://bedrock-runtime.{REGION}.amazonaws.com/openai/v1",
    api_key=provide_token(region=REGION),
    timeout=300,
)
bedrock = boto3.client("bedrock", region_name=REGION)


def answer_text(response) -> str:
    """The answer, without the \\confidence{N} marker Grok 4.7 sometimes ends it with."""
    return re.sub(r"\s*\\confidence\{\d+\}\s*$", "", response.output_text or "").strip()


def says_five_cents(text: str) -> bool:
    """True when the answer names five cents, as $0.05, .05, 5 cents or five cents."""
    amount = text.lower().replace("$", "").replace(",", "")
    # Lookarounds rather than \b, which does not match before the dot in ".05", and
    # which would let $1.05 or 0.055 through.
    return bool(re.search(r"(?<![\d.])0?\.05(?!\d)", amount)
                or re.search(r"\b(5|five)\s*cents?\b", amount))

## 1. First call

On `bedrock-runtime` name an inference profile as the model; the bare ID returns 400
`Invocation of model ID xai.grok-4.7 with on-demand throughput isn’t supported`. Grok
4.7 reasons before it answers and spends output tokens doing it, so too small a
`max_output_tokens` returns HTTP 200 with `status` `incomplete` and no text; the cells
here give it 2,000.

Asked for one word, the model sometimes adds a sentence of explanation, so the cell
checks the answer for `Canberra` rather than comparing the whole string. In October 2026
runs it also sometimes ended an answer with a `\confidence{100}` marker, which
`answer_text` in the setup cell removes.

Sampling does not use the OpenAI defaults. A request that sends neither parameter comes
back echoing `temperature` 0.7 and `top_p` 0.95, where the OpenAI API's own default for
both is 1, so set them explicitly if your output depends on them. The accepted ranges are
the OpenAI ones, `temperature` to 2 and `top_p` to 1, and the top of the temperature range
is not usable on a model that reasons on every request: at `temperature` 2.0 all eight
trials spent the whole 2,000-token budget on reasoning and returned no answer. At 1.5 the
same request answered in eight trials of nine, the ninth spending the budget the same way,
so check that the text is non-empty as well as that `status` is `completed` whatever you
set (October 2026).

Every call in this notebook passes `store=False`. The Responses API stores each response
for 30 days by default, and AWS advises passing `store=False` on every request you do
not want retained. With `global.`, a stored response can be kept in any commercial
Region the profile routes to, so use `us.` when data residency matters.

In [2]:
response = client.responses.create(
    model=GROK,
    input="What is the capital of Australia? Answer in one word.",
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(response)
print(answer)
print("says Canberra:", "canberra" in answer.lower())
print(f"status={response.status}  output_tokens={response.usage.output_tokens}")
if response.status != "completed":
    print("incomplete:", getattr(response.incomplete_details, "reason", None))

Canberra
says Canberra: True
status=completed  output_tokens=78


## 2. Streaming

With `stream=True` the SDK yields typed events, and answer text arrives in
`response.output_text.delta` events. The reasoning is encrypted, so no reasoning deltas
arrive. The `\confidence{100}` marker from section 1 can arrive in the last deltas, so a
UI that shows deltas as they arrive shows it too. The cell prints the raw deltas, then
the assembled answer from the final event with the marker removed by `answer_text`.

In [3]:
stream = client.responses.create(
    model=GROK,
    input="Count from 1 to 10, separated by commas.",
    max_output_tokens=2000,
    store=False,
    stream=True,
)
reasoning_deltas = 0
for event in stream:
    if event.type == "response.output_text.delta":
        print(event.delta, end="", flush=True)
    elif event.type in ("response.reasoning.delta", "response.reasoning_text.delta"):
        reasoning_deltas += 1  # count them here; a UI could show them as they arrive
    elif event.type in ("response.completed", "response.incomplete"):
        print(f"\nstatus={event.response.status}  reasoning deltas={reasoning_deltas}")
        print("answer:", answer_text(event.response))  # the assembled text, marker removed

1

,

2

,

3

,

... (10 outputs left out)

9

,

10


status=completed  reasoning deltas=0
answer: 1, 2, 3, 4, 5, 6, 7, 8, 9, 10


## 3. Multi-turn

The Responses API keeps no state between calls unless you ask it to. To hold a
conversation yourself, send it in `input`, including the model's output items as they
arrived. With `include=["reasoning.encrypted_content"]` the reasoning item carries an
encrypted blob that gives the next turn the model's earlier reasoning.

A turn sometimes arrives as a reasoning item with no message item at all, so send back
what `output` holds rather than assuming an answer is in it. Over eight repeats of this
first turn on `us.xai.grok-4.7` in `us-east-1` on 20 October 2026 every one carried a
message item, and one earlier call that day returned reasoning only. The turn after that
one still recalled the colour, because the conversation carries the reasoning item either
way.

In [4]:
messages = [{"role": "user", "content": "My favourite colour is teal. Reply with one word: noted."}]
first = client.responses.create(
    model=GROK, input=messages, include=["reasoning.encrypted_content"],
    max_output_tokens=2000, store=False,
)
messages += [item.model_dump(exclude_none=True) for item in first.output]  # reasoning and answer
messages.append({"role": "user", "content": "What is my favourite colour? One word."})
second = client.responses.create(
    model=GROK, input=messages, include=["reasoning.encrypted_content"],
    max_output_tokens=2000, store=False,
)

# A turn can come back as a reasoning item with no message, and an empty answer printed
# on its own reads as a failed call, so say which it was.
print("sent back:", [item.type for item in first.output])
print("turn 1   :", answer_text(first) or "(no message item: this turn returned reasoning only)")
print("turn 2   :", answer_text(second))
print("recalled teal:", "teal" in answer_text(second).lower())

sent back: ['reasoning', 'message']
turn 1   : noted
turn 2   : teal
recalled teal: True


With `store=True` the service keeps the response, and `previous_response_id` continues
from it, so the next request carries only the new turn. On `bedrock-runtime`, chaining
from a response made with `store=False` returns 400 `Previous response with id '…' not
found.`

`responses.retrieve` and `responses.delete` on `bedrock-runtime`'s `/openai/v1` answer
differently for different callers rather than at different times. In `us-east-1` on
2026-10-07 one identity got 200 from both over four stored responses on each of Grok 4.7
and Grok 4.6 at 07:03, 16 of 16 calls, and over five repeats at 07:39, 10 of 10, while a
second identity in the same account got 500 `internal_server_error` from both on 20 of 20
calls at 08:32, five repeats over `us.xai.grok-4.7` and five over `us.xai.grok-4.6`.
Waiting does not clear the 500, so read it as something about the caller rather than a
fault that passes, and look at the identity's permissions rather than retrying. The cell
stores one response, chains from it, then tries to delete it and reports whichever answer
it gets; its committed output comes from a run that got the 500 on the delete, in
`us-east-1` on 2 November 2026, where the `previous_response_id` turn above it in the
same cell still answered, so the 500 says something about reading back and deleting
rather than about storing. Prefer
`store=False` and hold the conversation yourself, as the cell above does; a response you
do store ages out after 30 days.

In [5]:
turn1 = client.responses.create(
    model=GROK,
    input="My favourite colour is teal. Reply with one word: noted.",
    max_output_tokens=2000,
    store=True,  # kept for 30 days unless deleted, so turn 2 can refer to it
)
try:
    turn2 = client.responses.create(
        model=GROK,
        input="What is my favourite colour? One word.",
        previous_response_id=turn1.id,
        max_output_tokens=2000,
        store=False,
    )
    answer = answer_text(turn2)
    print("turn 2:", answer)
    print("recalled teal:", "teal" in answer.lower())
finally:
    # The delete runs even when the call above raises, because a response this cell
    # stored and did not delete is kept for 30 days. Both 200 and 500 came back from this
    # delete on 2026-10-07, so report whichever arrives rather than letting it stop the
    # notebook or replace the error on its way out.
    try:
        client.responses.delete(turn1.id)
        print("deleted the stored response")
    except openai.APIStatusError as exc:
        print(f"delete refused: {exc.status_code} {exc.code}; the response stays stored")

turn 2: teal
recalled teal: True


delete refused: 500 internal_server_error; the response stays stored


## 4. Reasoning

Grok 4.7 always reasons. The `effort` in `reasoning` can be `low`, `medium`, `high` or
`xhigh`, and a response echoes the effort it used. The [model
card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-xai-grok-4-7.html)
gives `high` as the default (card read 29 October 2026). The last line of the cell below
sends no `reasoning` field at all, which shows the effort the service applies when the
field is left out, and that is worth checking per model: the Grok 4.6 card gives `low`
as its default where that service returned `high` on 3 of 3 such calls
([`02-grok-4-6`](02-grok-4-6.ipynb)). Here the card and the service agree, `high` on 3
of 3 (`us-east-1`, 29 October 2026), so set `effort` explicitly if you want a different
one. `none`, `minimal` and `max` each returned 400 `Unsupported value` on this API, so
reasoning cannot be switched off here; Chat Completions takes `minimal` (section 8). The
Things that differ table has the error text, which does not agree with itself about which
values are supported.

The reasoning is encrypted: the `reasoning` item holds `encrypted_content` and no
readable text. `usage.output_tokens_details.reasoning_tokens` counts the reasoning you
pay for.

In [6]:
response = client.responses.create(
    model=GROK,
    input='A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.',
    reasoning={"effort": "medium"},
    include=["reasoning.encrypted_content"],
    max_output_tokens=4000,
    store=False,
)
reasoning = [item for item in response.output if item.type == "reasoning"]
answer = answer_text(response)
print("answer      :", answer)
print("says $0.05  :", says_five_cents(answer))  # the answer can run long, or word the amount
print("items       :", [item.type for item in response.output])
blob = "".join(item.encrypted_content or "" for item in reasoning)
readable = sum(len(item.content or []) + len(item.summary or []) for item in reasoning)
print(f"reasoning   : {len(blob)} characters of encrypted_content, {readable} readable parts")
print("effort used :", response.reasoning.effort)  # echoed back by the service
print(f"reasoning_tokens={response.usage.output_tokens_details.reasoning_tokens}  status={response.status}")

# The card gives `high` as the default. This checks what the service does, because the
# Grok 4.6 card gives `low` and that service returns `high`.
unset = client.responses.create(
    model=GROK, input="Reply with one word: noted.", max_output_tokens=2000, store=False
)
print("effort unset:", unset.reasoning.effort)

answer      : $0.05
says $0.05  : True
items       : ['reasoning', 'message']
reasoning   : 4501 characters of encrypted_content, 0 readable parts
effort used : medium
reasoning_tokens=267  status=completed


effort unset: high


## 5. Tool use

A Responses tool is flat: `name`, `description` and `parameters` sit at the top level,
where Chat Completions nests them under `function`. Each call arrives as a
`function_call` output item. Run it, send the call and a `function_call_output` item
with the same `call_id` back in `input`, and repeat until the model answers in text.
Check each call's arguments before running it: a call says the model wants a tool, and
says nothing about whether the arguments are usable.

In [7]:
def lookup_inventory(sku: str) -> dict:
    """Stand-in for a real inventory service."""
    stock = {"A-100": 42, "B-200": 0}
    if sku.upper() not in stock:
        return {"sku": sku, "error": "unknown SKU"}
    return {"sku": sku, "quantity": stock[sku.upper()]}


TOOLS = [{
    "type": "function",
    "name": "lookup_inventory",
    "description": "Current stock level for a SKU.",
    "parameters": {
        "type": "object",
        "properties": {"sku": {"type": "string", "description": "SKU code, e.g. A-100"}},
        "required": ["sku"],
    },
}]
IMPLEMENTATIONS = {"lookup_inventory": lookup_inventory}


def run_tool(name: str, args: dict) -> dict:
    """Refuse a call that names an unknown tool or does not match the tool's parameters."""
    spec = next((t for t in TOOLS if t["name"] == name), None)
    if spec is None:
        return {"error": f"unknown tool {name}"}
    params = spec["parameters"]
    missing = [k for k in params["required"] if k not in args]
    unknown = [k for k in args if k not in params["properties"]]
    wrong_type = [k for k in args if k in params["properties"] and params["properties"][k].get("type") == "string"
                  and not isinstance(args[k], str)]
    if missing or unknown or wrong_type:
        return {"error": f"bad arguments: missing {missing}, unknown {unknown}, not a string {wrong_type}"}
    return IMPLEMENTATIONS[name](**args)

In [8]:
items = [{"role": "user", "content": "How many units of SKU A-100 do we have?"}]
for _ in range(5):  # bounded, so a model that keeps calling tools cannot loop forever
    response = client.responses.create(
        model=GROK, input=items, tools=TOOLS, max_output_tokens=2000, store=False
    )
    calls = [item for item in response.output if item.type == "function_call"]
    if not calls:
        break
    for call in calls:
        args = parse_json_lenient(call.arguments)
        result = run_tool(call.name, args)
        print(f"tool call: {call.name}({args}) -> {result}")
        items.append({"type": "function_call", "call_id": call.call_id, "name": call.name,
                      "arguments": call.arguments})
        items.append({"type": "function_call_output", "call_id": call.call_id, "output": json.dumps(result)})
else:
    raise RuntimeError("no answer after 5 rounds of tool calls")

answer = answer_text(response)
print("answer:", answer)
# 42 is the quantity the stand-in tool reports for A-100, so the answer must carry it.
# The loop breaks on any response with no function_call item, a reply cut off at the
# output cap included, so print status beside the verdict: "incomplete" marks an
# answer that the cap ended, which would otherwise read as a complete one.
print("42 units reported:", "42" in answer, " status:", response.status)

tool call: lookup_inventory({'sku': 'A-100'}) -> {'sku': 'A-100', 'quantity': 42}


answer: **42 units** of SKU A-100 are currently in stock.
42 units reported: True  status: completed


## 6. Structured output

`text.format` with a JSON Schema asks for output in that shape. Check `status` first,
because a cut-off answer is cut-off JSON, and parse leniently. A schema controls the
shape of the output and has no say over the values in it, so the cell checks each value
against the source text.

In [9]:
SCHEMA = {
    "type": "object",
    "properties": {
        "invoice_id": {"type": "string"},
        "vendor": {"type": "string"},
        "total": {"type": "number"},
        "currency": {"type": "string"},
    },
    "required": ["invoice_id", "vendor", "total", "currency"],
    "additionalProperties": False,
}
SOURCE = "Invoice: INV-1042\nVendor: Acme Pty Ltd\nAmount due: AUD 1,280.50\nDue date: 30 October 2026"
EXPECTED = {"invoice_id": "INV-1042", "vendor": "Acme Pty Ltd", "total": 1280.5, "currency": "AUD"}

response = client.responses.create(
    model=GROK,
    input=("Extract invoice_id, vendor, total and currency from this invoice as JSON. Write the "
           f"total as a number without a thousands separator, so 2,345.60 becomes 2345.6.\n\n{SOURCE}"),
    text={"format": {"type": "json_schema", "name": "invoice", "strict": True, "schema": SCHEMA}},
    max_output_tokens=2000,
    store=False,
)
if response.status != "completed":
    raise RuntimeError(f"incomplete output ({getattr(response.incomplete_details, 'reason', response.status)})")

invoice = parse_json_lenient(response.output_text)
print(json.dumps(invoice, indent=2))
wrong = {k: invoice.get(k) for k, v in EXPECTED.items() if invoice.get(k) != v}
print("values match the source" if not wrong else f"values that do NOT match the source: {wrong}")

{
  "invoice_id": "INV-1042",
  "vendor": "Acme Pty Ltd",
  "total": 1280.5,
  "currency": "AUD"
}
values match the source


## 7. Vision

Send the image as a data URL in an `input_image` part, before the text part. The slide
comes from a public AWS talk and ships with this repository, so the answer is known: its
title is `Ingestion from database`, and the cell counts how many of its three callout
lines the model quotes.

In [10]:
from bedrock import SLIDE_CALLOUTS, SLIDE_TITLE, keyword_recall, slide_data_url

response = client.responses.create(
    model=GROK,
    input=[{"role": "user", "content": [
        {"type": "input_image", "image_url": slide_data_url()},
        {"type": "input_text", "text": "Read this slide. Give its title, then quote the three green callout lines."},
    ]}],
    max_output_tokens=2000,
    store=False,
)
answer = answer_text(response)
hits, total = keyword_recall(answer, SLIDE_CALLOUTS)
print(answer)
print(f"\nstatus={response.status}")
print(f"title found: {SLIDE_TITLE.lower() in answer.lower()}   callouts found: {hits}/{total}")

Title: Ingestion from database

"✓ Pay according to job duration"
"✓ Lower compute cost, up to 90%"
"✓ Lower storage cost for rarely accessed data"

status=completed
title found: True   callouts found: 3/3


## 8. Chat Completions on the same path

Grok 4.7 also serves Chat Completions on the same path, where the reasoning effort is
`reasoning_effort` rather than a nested object. The [model
card](https://docs.aws.amazon.com/bedrock/latest/userguide/model-card-xai-grok-4-7.html)
says "The Chat Completions API does not return reasoning tokens" (read 22 October 2026),
and that holds for the reasoning text: the message carried no `reasoning` field at all on
3 of 3 calls at `reasoning_effort="high"` in `us-east-1` on 22 October 2026, and the cell
below defaults that field, so an absent one prints 0 characters. It does not hold for the
count. `usage.completion_tokens_details.reasoning_tokens` was non-zero on each of those
calls, 635, 461 and 618 against the 287 committed below, and 294 on one call that sent no
effort field. The count is the only thing here that tells you what the reasoning cost,
because there is no readable trace on either API (section 4).

The efforts this API accepts are not the ones the Responses API accepts: `minimal` is
taken here, 200 on 4 of 4 calls spending 67 to 90 reasoning tokens, and refused there
(Things that differ).

In [11]:
response = client.chat.completions.create(
    model=GROK,
    messages=[{"role": "user", "content": 'A bat and a ball cost $1.10 in total. The bat costs $1.00 more than the ball. How much does the ball cost? Answer with the amount only.'}],
    reasoning_effort="high",
    max_completion_tokens=4000,
)
message = response.choices[0].message
details = response.usage.completion_tokens_details
answer = (message.content or "").strip()
print("answer           :", answer)
print("says $0.05       :", says_five_cents(answer))
print("message.reasoning:", f"{len(getattr(message, 'reasoning', None) or '')} characters")
print("reasoning_tokens :", getattr(details, "reasoning_tokens", None))
print("finish_reason    :", response.choices[0].finish_reason)

answer           : $0.05
says $0.05       : True
message.reasoning: 0 characters
reasoning_tokens : 287
finish_reason    : stop


## 9. On `bedrock-runtime`: Converse

The same profile IDs work with Converse through boto3, which signs each request with
your AWS credentials. The reasoning arrives as a `reasoningContent` block holding
`redactedContent`, the Converse form of the encrypted trace, so read the content blocks
by type rather than by index.

Passing the reasoning effort through `additionalModelRequestFields` made no measured
difference here. On `us.xai.grok-4.7` in `us-east-1` on 14 October 2026 the call returned
HTTP 200 for `{"reasoning_effort": "low"}`, `{"reasoningEffort": "xhigh"}` and
`{"reasoning": {"effort": "xhigh"}}`, and equally for `{"reasoning_effort": "none"}`,
which the Responses path rejects with 400 `Unsupported value: 'none' is not supported
with the 'us.xai.grok-4.7' model.`, for `{"reasoning_effort": "bogus"}` and for an
invented key, so a 200 here does not tell you that the field applied. No reply carried
`additionalModelResponseFields`. The spend did not separate either: on one reasoning
question with `maxTokens` 8,000, four calls at `low` spent 320 to 770 output tokens and
four at `xhigh` spent 409 to 732, against 521 and 990 for the same question with no field
at all. The same question on the Responses API, same model and budget, spent 216 to 323
reasoning tokens at `low` and 698 to 777 at `xhigh` over three calls each, and echoed the
effort it used. Set effort on a path that echoes it back (section 4), and read what a
Converse call spent from `usage` rather than from what the request asked for.

Converse takes a service tier too, under a different name and with a different answer.
The field is `serviceTier={"type": ...}`, the enum botocore 1.43.59 will send is
`default`, `flex`, `priority` and `reserved` with no `auto` in it, and the model card
restricts Priority and Flex to the Responses, Chat Completions and Invoke APIs for this
model. The cell checks that instead of taking it on trust: on `us.xai.grok-4.7` in
`us-east-1` on 19 October 2026, `default` was served and echoed back, `flex` and
`priority` both returned `ValidationException: The provided service tier is not supported
for this model.`, and `reserved`, which is in the enum, returned `ValidationException: The
provided model identifier is invalid.` rather than anything about the tier. Keep Flex and
Priority work on the Responses or Chat Completions path (section 8), or on Invoke, which
served both on 2 November 2026 (section 10). A Converse reply
carries `serviceTier` back only when the request asked for one; omitting the field left it
absent in that run.

In [12]:
runtime = boto3.client("bedrock-runtime", region_name=REGION)
response = runtime.converse(
    modelId=GROK,
    messages=[{"role": "user", "content": [{"text": "What is the capital of Australia? Answer in one word."}]}],
    inferenceConfig={"maxTokens": 2000},
)
blocks = response["output"]["message"]["content"]
print("blocks    :", [next(iter(b)) for b in blocks])
print("reasoning :", [list(b["reasoningContent"]) for b in blocks if "reasoningContent" in b])
text = "".join(b["text"] for b in blocks if "text" in b)
text = re.sub(r"\s*\\confidence\{\d+\}\s*$", "", text).strip()  # the same marker as answer_text
print("answer    :", text)
print("says Canberra:", "canberra" in text.lower())
print("stopReason:", response["stopReason"])
# A reply carries serviceTier only when the request asked for one, as this one did not.
print("serviceTier in reply:", "serviceTier" in response)

# The card restricts Priority and Flex to the Responses, Chat Completions and Invoke APIs
# for this model, so ask Converse for each tier and print what it served.
tiers = {}
for tier in ("default", "flex", "priority", "reserved"):
    try:
        reply = runtime.converse(
            modelId=GROK,
            messages=[{"role": "user", "content": [{"text": "Reply with one word: noted."}]}],
            inferenceConfig={"maxTokens": 2000},
            serviceTier={"type": tier},
        )
        tiers[tier] = reply["serviceTier"]["type"]
        print(f"tier {tier:9} served as {reply['serviceTier']['type']}")
    except runtime.exceptions.ValidationException as exc:
        tiers[tier] = "refused"
        print(f"tier {tier:9} refused   {exc.response['Error']['Message']}")

# Written out longhand from what the prose above claims, so that a change in what Converse
# serves prints False rather than agreeing with whatever came back.
enum = runtime.meta.service_model.shape_for("ServiceTierType").enum  # no `auto` in it
print("Converse serves Standard only:",
      tiers == {"default": "default", "flex": "refused", "priority": "refused",
                "reserved": "refused"} and "auto" not in enum)

blocks    : ['reasoningContent', 'text']
reasoning : [['redactedContent']]
answer    : Canberra
says Canberra: True
stopReason: end_turn
serviceTier in reply: False


tier default   served as default
tier flex      refused   The provided service tier is not supported for this model.
tier priority  refused   The provided service tier is not supported for this model.


tier reserved  refused   The provided model identifier is invalid.
Converse serves Standard only: True


## 10. On `bedrock-runtime`: InvokeModel

The card ticks a fourth API, Invoke, and its Sample Code section carries no example for
it (card read 2 November 2026), so the body shape below comes from measurement rather
than from the card. InvokeModel takes the Chat Completions body serialised into `body`,
rather than Converse's content blocks and `inferenceConfig`: `messages`, `max_tokens`
and `reasoning_effort` go in under the names Chat Completions uses (section 8), and the
reply is a `chat.completion` object read through `choices[0].message` and a
`finish_reason`, rather than an `output` message with a `stopReason`. Name the inference
profile here, as the other calls on this endpoint do; the bare `xai.grok-4.7` returns
`ValidationException: Invocation of model ID xai.grok-4.7 with on-demand throughput
isn’t supported.`

Where the reasoning goes is what changes on this route. There is no trace to read and no
`reasoning` field to look in: `message` carried `annotations`, `content`, `refusal` and
`role` and nothing else on 14 of 14 calls over both Grok profiles, while
`usage.completion_tokens_details.reasoning_tokens` counted the reasoning on each of
them. On `us.xai.grok-4.7` in `us-east-1` on 2 November 2026 the one-word question below
spent 63, 68 and 82 reasoning tokens over three calls at `high`, 82 again on the call
committed below, and 78 on a call that sent no effort field. Chat Completions behaves
the same way on this family (section 8), and other families do not: Nemotron returns its
working inline in `content` behind a closing tag
([`10-nvidia-nemotron/01`](../10-nvidia-nemotron/01-nemotron-nano-and-super.ipynb)).

Two things this route carries that Converse does not. It echoes `service_tier` back, and
it serves Flex and Priority, which is what the card says for this model: `flex` and
`priority` each came back echoed on `us.xai.grok-4.7` in `us-east-1` on 2 November 2026,
on a path where Converse refuses both (section 9). The card also gives application
inference profiles to the Invoke and Converse APIs and not to Responses or Chat
Completions, so this is the route to take when you want inference cost tagged per
application.

`invoke_model_with_response_stream` takes the same body. It answered on
`us.xai.grok-4.7` in `us-east-1` on 2 November 2026, delivering `chat.completion.chunk`
objects with the text in `choices[0].delta.content`. How many events carry a reply is
not fixed: a count from 1 to 5 arrived in 11 to 16 events over eight runs across the
two Grok profiles on 2 and 3 November 2026, so assemble the deltas rather than
counting events.
Section 2 streams this model on the Responses API, which is the path the rest of this
notebook uses.

In [13]:
CAPITAL = "What is the capital of Australia? Answer in one word."

# The Chat Completions body goes into `body` serialised, so the field names are the Chat
# Completions ones rather than Converse's inferenceConfig.
body = {
    "messages": [{"role": "user", "content": CAPITAL}],
    "max_tokens": 2000,
    "reasoning_effort": "high",
}
response = runtime.invoke_model(modelId=GROK, body=json.dumps(body))
payload = json.loads(response["body"].read())
choice = payload["choices"][0]

# Raise before reading the text. A budget that ran out leaves a short or empty `content`
# that prints here exactly like a finished answer, and the verdict below would read it as
# a wrong answer rather than as a call that never got to answer.
if choice["finish_reason"] != "stop":
    raise RuntimeError(f"finish_reason={choice['finish_reason']}, so the text is cut: "
                       "raise max_tokens")

content = choice["message"].get("content") or ""
answer = re.sub(r"\s*\\confidence\{\d+\}\s*$", "", content).strip()  # as answer_text does
print("object       :", payload["object"], "  service_tier:", payload.get("service_tier"))
print("message keys :", sorted(choice["message"]))  # no `reasoning` among them
print("reasoning    :",
      payload["usage"]["completion_tokens_details"]["reasoning_tokens"],
      "tokens counted, with no trace to read")
print("answer       :", answer)
print("says Canberra:", "canberra" in answer.lower())

object       : chat.completion   service_tier: default
message keys : ['annotations', 'content', 'refusal', 'role']
reasoning    : 82 tokens counted, with no trace to read
answer       : Canberra
says Canberra: True


## 11. IAM permissions

Calling through an inference profile needs `bedrock:InvokeModel` on the profile and on
the foundation model in each Region the profile routes to. For Grok 4.7 on
`bedrock-runtime` the model card adds your account's default project,
`arn:aws:bedrock:{region}:{account-id}:project/default`. The cell builds that policy
from the profile's own ARNs and prints it with the account ID replaced.

In [14]:
account = boto3.client("sts").get_caller_identity()["Account"]
profile = bedrock.get_inference_profile(inferenceProfileIdentifier=GROK)

POLICY = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Action": ["bedrock:InvokeModel", "bedrock:InvokeModelWithResponseStream"],
        "Resource": [
            profile["inferenceProfileArn"],
            *(m["modelArn"] for m in profile["models"]),  # each Region the profile routes to
            f"arn:aws:bedrock:{REGION}:{account}:project/default",
        ],
    }],
}
safe_print(json.dumps(POLICY, indent=2))  # replaces the account ID before printing

{
  "Version": "2012-10-17",
  "Statement": [
    {
      "Effect": "Allow",
      "Action": [
        "bedrock:InvokeModel",
        "bedrock:InvokeModelWithResponseStream"
      ],
      "Resource": [
        "arn:aws:bedrock:us-east-1:123456789012:inference-profile/us.xai.grok-4.7",
        "arn:aws:bedrock:us-east-1::foundation-model/xai.grok-4.7",
        "arn:aws:bedrock:us-east-2::foundation-model/xai.grok-4.7",
        "arn:aws:bedrock:us-west-2::foundation-model/xai.grok-4.7",
        "arn:aws:bedrock:us-east-1:123456789012:project/default"
      ]
    }
  ]
}


## 12. Prompt caching

Caching on this family is implicit. No request parameter turns it on, and there are no
cache blocks to place: the service keys on the start of the request, so stable content
goes at the front and the question at the end.
`usage.input_tokens_details.cached_tokens` reports how much of the input was served from
cache, and `cache_write_tokens` stays 0, so filling the cache costs nothing extra,
unlike the explicit `cache_control` blocks Claude takes
([`02-anthropic-claude/02`](../02-anthropic-claude/02-thinking-tools-and-caching.ipynb)).
The card's feature table for this endpoint lists implicit prompt caching as supported
and explicit prompt caching as not supported, and prices a cache read at a quarter of
the input rate on each of the three inference options it prices: $0.55 against $2.20 for
Geo and $0.50 against $2.00 for Global in the commercial Regions, and $0.66 against
$2.64 for Geo in GovCloud, where the card's separate AWS GovCloud (US) table puts each
of its three rates 20% above the commercial Geo one (card read 7 November 2026). Geo is
the option the card gives in GovCloud, so a reader there budgets from that third row
rather than from the commercial Geo one. All three rows are Standard-tier rates, which
the card multiplies by 1.75 for Priority and by 0.5 for Flex.

The cell sends one 18,000-token catalogue up to six times, asking a different question
each time, and stops at the first send that reports cached tokens. On `us.xai.grok-4.7`
in `us-east-1` on 10 October 2026 that was send 2, with 18,048 of the request's 18,167
input tokens served from cache. The run committed below also hit on send 2, on 2
November 2026, and reported the whole 18,180-token input cached rather than the prefix
alone, so how much of a request the cache covers is not fixed. Grok 4.3 on
`bedrock-mantle` does not hold a cache as reliably, which
[`01-grok-4-3`](01-grok-4-3.ipynb) measures.

In [15]:
import uuid

# Implicit caching keys on the start of the request, so the catalogue goes first and the
# question last. Nothing in the request asks for it. The run id makes the prefix new on
# every run, so send 1 cannot hit a cache an earlier run filled.
def entry(sku: int) -> tuple[int, float]:
    """Lead time in weeks and unit price, from values that do not run in a sequence a
    model could extrapolate instead of reading."""
    return sku * 13 % 7 + 2, (sku * 37) % 900 + 9.99

CATALOGUE = "\n".join(
    f"SKU A-{n} | lead time {entry(n)[0]} weeks | unit price ${entry(n)[1]:.2f}"
    for n in range(100, 1100)
)
PREFIX = f"Reference catalogue {uuid.uuid4()}\n{CATALOGUE}"
LOOKUPS = [(137, "unit price"), (242, "lead time"), (908, "unit price"),
           (451, "lead time"), (608, "unit price"), (771, "lead time")]

hit_on = None
for send, (sku, field) in enumerate(LOOKUPS, start=1):
    weeks, price = entry(sku)
    expected = f"{price:.2f}" if field == "unit price" else f"{weeks}"
    response = client.responses.create(
        model=GROK,
        input=f"{PREFIX}\n\nQuestion: What is the {field} of SKU A-{sku}? Answer with the value only.",
        reasoning={"effort": "low"},  # this model refuses "none" (section 4)
        max_output_tokens=500,
        store=False,
    )
    usage, details = response.usage, response.usage.input_tokens_details
    answer = answer_text(response)
    print(f"send {send}: A-{sku} {field}  input_tokens={usage.input_tokens} "
          f"cached={details.cached_tokens} written={details.cache_write_tokens} "
          f"status={response.status}")
    print(f"         answer={answer!r}, holds {expected}: {expected in answer}")
    if details.cached_tokens:
        hit_on = send
        break
print("first send served from cache:", hit_on or f"none of {len(LOOKUPS)}")

send 1: A-137 unit price  input_tokens=18180 cached=0 written=0 status=completed
         answer='$578.99', holds 578.99: True


send 2: A-242 lead time  input_tokens=18180 cached=18180 written=0 status=completed
         answer='5 weeks', holds 5: True
first send served from cache: 2


## Things that differ

What trips people up on this family, and what to do about it.

| Behaviour | What you see | What to do |
|---|---|---|
| Reading the answer | `output[0]` is a reasoning item on every call, because the model always reasons | Read `output_text`, or pick output items by `type`. |
| Stored by default | `store` defaults to true, and the response is kept for 30 days | Pass `store=False` unless you will chain from the response. Deleting can be refused here, as the next row says. |
| Reading back or deleting a stored response | `responses.retrieve` and `responses.delete` on `bedrock-runtime`'s `/openai/v1` follow the calling identity, not the clock: on 2026-10-07 in `us-east-1` one identity got 200 on 26 of 26 calls at 07:03 and 07:39, and a second got 500 `internal_server_error` on 20 of 20 at 08:32, over both Grok profiles, and runs on 19 October and 1 and 2 November 2026 got the 500 on the delete with the `previous_response_id` turn in the same cell answering | Prefer `store=False` and keep the conversation yourself (section 3). Where you do store, handle both answers, and look at the identity's permissions rather than retrying a 500. |
| Answer cut short | HTTP 200 with `status` `incomplete` and `incomplete_details.reason` `max_output_tokens` | Raise `max_output_tokens`, and check `status` before using the text. |
| Structured output values | Valid JSON that holds a wrong value | A schema only controls the shape. Check the values you depend on. |
| Rate limits | 429 under load | Retry with backoff ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Prompt caching | Implicit: nothing in the request turns it on, `usage.input_tokens_details.cached_tokens` reports the hit, and `cache_write_tokens` stays 0. Send 2 of an 18,000-token prefix had 18,048 of 18,167 input tokens served from cache on 10 October 2026, and the whole 18,180 on 2 November 2026 (`us.xai.grok-4.7`, `us-east-1`) | Put stable content at the front of the request, and read `cached_tokens` to confirm the discount before you count on it (section 12). |
| Bare model ID on `bedrock-runtime` | 400 `Invocation of model ID xai.grok-4.7 with on-demand throughput isn’t supported` | Name `us.xai.grok-4.7` or `global.xai.grok-4.7`. |
| Where the bare ID is the one to use | The card's `bedrock-runtime` feature table gives Prompt optimization and Model evaluation to "base model ID `xai.grok-4.7` only", and adds for model evaluation that "inference parameters are not supported". Neither feature appears on the Grok 4.6 card's table for that endpoint, and Grok 4.3 is not served on `bedrock-runtime` at all ([model cards](https://docs.aws.amazon.com/bedrock/latest/userguide/model-cards.html) read 7 November 2026) | Name a profile for inference, as the row above says, and the bare `xai.grok-4.7` where you set up prompt optimization or a model evaluation job. |
| `us.` outside the US geo | 400 `The provided model identifier is invalid.` (measured from `eu-central-1`, 18 October 2026); from `ca-central-1` and `ca-west-1` the same ID answered 200, because the card lists both as source Regions for the US geo | Check the card's source-Region table for the geo you call, or call `GetInferenceProfile` from that Region as the card suggests, before falling back to `global.xai.grok-4.7`: `global.` routes to any commercial Region, so prefer `us.` where residency matters. |
| Not on `bedrock-mantle` | 404 `The model 'xai.grok-4.7' does not exist`, in `us-east-1` and `us-west-2` (October 2026) | Call `bedrock-runtime`. Grok 4.6 is the Grok on `bedrock-mantle` ([`02-grok-4-6`](02-grok-4-6.ipynb)). |
| Reasoning on Chat Completions | The model card says the Chat Completions API "does not return reasoning tokens". The message carries no `reasoning` field on 3 of 3 calls at `reasoning_effort="high"`, and `usage.completion_tokens_details.reasoning_tokens` is non-zero on those same calls: 635, 461 and 618, with 294 on one call that sent no effort field (`us-east-1`, 22 October 2026) | Read `reasoning_tokens` to see what the reasoning cost. There is no readable trace to show a user on either API (section 8). |
| Where the reasoning trace arrives | It goes by route, and two of the four routes carry no trace: `reasoning` items holding `encrypted_content` on Responses (section 4), a `reasoningContent` block holding `redactedContent` on Converse (section 9), and nothing readable on Chat Completions (section 8) or on Invoke, where `message` carried no `reasoning` field on 14 of 14 calls and the spend showed only in `usage.completion_tokens_details.reasoning_tokens` (`us-east-1`, 2 November 2026) | Read the trace by route, and where there is no trace, read `reasoning_tokens` for what it cost. A reader written for one route does not carry over to another (section 10). |
| Reasoning effort values | The two APIs take different sets, and the errors disagree about them. `none` and `max` are refused on both (3 of 3 calls each), and `minimal` is refused on the Responses API and accepted on Chat Completions, 200 on 4 of 4 calls spending 67 to 90 reasoning tokens. On the Responses API the refusals of `none` and of `max` both say *"Supported values are: 'low', 'medium', 'high', and 'xhigh'"* while the refusal of `minimal` also lists `none` and `max`; on Chat Completions `max` returns a shorter 400 that lists nothing, `Unsupported value: 'max' is not supported with this model.` An unknown value is a different error again on each, `Invalid value: 'bogus'.`, with `minimal` in the list on Chat Completions and absent from it on Responses (`us-east-1`, 22 October 2026) | Use `low`, `medium`, `high` or `xhigh`, which both APIs accept, and do not take the supported-value list in the error as authoritative. |
| Reasoning effort on Converse | `additionalModelRequestFields` returned HTTP 200 for each shape tried, including `{"reasoning_effort": "none"}`, which the Responses path rejects with 400, and an invented key; no reply carried `additionalModelResponseFields`, and output-token spend did not separate `low` from `xhigh` over four calls each (`us.xai.grok-4.7`, `us-east-1`, 14 October 2026) | Set effort on the Responses API or on Chat Completions, where the service echoes what it used (section 4), and read what a Converse call spent from `usage` (section 9). |
| Sampling defaults and range | Not the OpenAI ones: a request that sends neither parameter comes back echoing `temperature` 0.7 and `top_p` 0.95, measured on `us.xai.grok-4.7` in `us-east-1` and on `global.xai.grok-4.7` in `us-east-1` and `eu-central-1` (October 2026). Sending one of the two leaves the other at the model's default. `temperature` above 2 returns 400 `Invalid 'temperature': decimal above maximum value. Expected a value <= 2, but got 3.0 instead.` and `top_p` above 1 the same error with a maximum of 1, and `temperature` 2.0 is inside the range and still returned no answer in eight trials of eight, each with `status` `incomplete`, `incomplete_details.reason` `max_output_tokens` and the whole 2,000-token budget spent. At 1.5 the same request answered in eight trials of nine | Set both explicitly if your output depends on them, keep `temperature` at or below 1.5, and check that the text is non-empty as well as that `status` is `completed`. Code ported from another OpenAI-compatible model samples differently here (section 1). |
| Default effort | A request with no `reasoning.effort` comes back with `reasoning.effort` `high`, which is what the model card gives as the default (card read 29 October 2026; `high` on 3 of 3 calls in `us-east-1` that day). The field is worth checking per model: the Grok 4.6 card gives `low` and that service returned `high` on 3 of 3 such calls the same day ([`02-grok-4-6`](02-grok-4-6.ipynb)) | Set `effort` explicitly when you care what you pay for reasoning. |
| Background mode | 400 `The background parameter is not supported.` | There is no `bedrock-mantle` route for this model, so background mode is not available for it. Poll your own job instead. |
| Web search | 400 ``Tool type 'web_search' is not supported for model `xai.grok-4.7`.`` | Search with your own tool and pass the results in. |
| Answer marker | An answer that ends in `\confidence{100}` | Remove it before display, as `answer_text` does. |
| Application inference profiles | Supported for the Invoke and Converse APIs, and not for Responses or Chat Completions, per the model card | Tag cost by application profile on the Invoke (section 10) or Converse (section 9) path, and carry your own attribution on the Responses calls the rest of this notebook uses. |
| Service tiers | On Responses and Chat Completions, `service_tier` takes `auto`, `default`, `flex` and `priority`. Measured on `us.xai.grok-4.7` and `us.xai.grok-4.6` in `us-east-1` in October 2026, `auto` and `default` both came back echoed as `default`, as did omitting the field, and `flex` and `priority` echoed themselves. A tier name from the model card is not necessarily a value: `standard` returns 400 `service tier provided is invalid. Please check the documentation for valid service tiers`, and `reserved` returns 400 `Invalid value: 'reserved'. Supported values are: 'auto', 'default', 'flex', and 'priority'.` Converse is different again: the field is `serviceTier={"type": ...}`, `auto` is not in its enum, and this model serves Standard only there, with `flex` and `priority` both returning `ValidationException: The provided service tier is not supported for this model.` (`us.xai.grok-4.7`, `us-east-1`, 19 October 2026), as the card states | Send `default`, or omit the field, for Standard, and `priority` or `flex` for the others. Do not pass the card's display names. Keep Flex and Priority work on the Responses, Chat Completions or Invoke path (section 10), since Converse (section 9) serves neither ([`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)). |
| Permissions | The profile alone is not enough on `bedrock-runtime`, per the model card | Allow the foundation models and your default project too (section 11). |

## Next

- Grok 4.6, on both endpoints: [`02-grok-4-6`](02-grok-4-6.ipynb)
- Grok 4.3, on `bedrock-mantle`: [`01-grok-4-3`](01-grok-4-3.ipynb)
- Converse and inference profiles in depth:
  [`00-foundations/04`](../00-foundations/04-bedrock-runtime-converse-and-profiles.ipynb)
- Other families on the Responses API:
  [`03-google-gemma/01`](../03-google-gemma/01-gemma4-end-to-end.ipynb) (Gemma 4),
  [`01-openai-gpt/01`](../01-openai-gpt/01-responses-api-core.ipynb) (OpenAI GPT)
- Choosing a model and API across families:
  [`99-cross-cutting/01`](../99-cross-cutting/01-choosing-a-model-and-api.ipynb)
- Quotas, retries and latency:
  [`00-foundations/03`](../00-foundations/03-scaling-tiers-and-latency.ipynb)
- Production hardening:
  [`99-cross-cutting/03`](../99-cross-cutting/03-production-hardening.ipynb)